In [ ]:
%run "00_globals_and_db.ipynb"


In [ ]:
# Import a specialized tool for fetching web pages/files (scrapling)
from scrapling.fetchers import FetcherSession

# 2. LOAD DATA 
# Read a JSON Lines file (each line is a dictionary) and split it into a list
lines = (DIR_RAW / "case_rows.jsonl").read_text(encoding="utf-8").splitlines()

In [ ]:
# Establish a connection to the database and create a 'cursor' to execute SQL commands
cnx = db_connect()
cur = cnx.cursor()

ok = 0
fail = 0

# START THE DOWNLOAD PROCESS
# Use a 'FetcherSession' which handles the connection details (90-second timeout, 3 retries)
with FetcherSession(timeout=90, retries=3) as session:
    for line in lines:
        # Convert the text line into a Python dictionary (per row)
        row = json.loads(line)
        row_id = int(row["row_id"])
        pdf_url = row["pdf_url"]
        pdf_url_hash = row["pdf_url_hash"]

        # SKIP LOGIC: Don't download if we already have this URL hash in our database
        db_exec(cur, "SELECT pdf_id FROM raw_pdf WHERE pdf_url_hash=%s LIMIT 1", (pdf_url_hash,))
        if cur.fetchone():
            continue
        # ATTEMPT THE DOWNLOAD
        resp = None
        try:
            # Request the file from the web
            resp = session.get(pdf_url)
            
            # Check if the download failed (not status 200) or if the file is suspiciously small (< 2KB)
            if resp.status != 200 or not resp.body or len(resp.body) < 2000:
                raise RuntimeError(f"bad status={resp.status} size={len(resp.body) if resp.body else 0}")

            # Create a unique filename based on the file's actual content (a SHA256 hash)
            sha = sha256_bytes(resp.body)
            out_path = DIR_PDFS / f"{sha}.pdf"
            
            # Save the file to the hard drive if it doesn't exist yet
            if not out_path.exists():
                out_path.write_bytes(resp.body)

            # Record the successful download in the 'raw_pdf' table
            upsert_raw_pdf(cur, row_id, pdf_url, pdf_url_hash, sha, str(out_path), resp.status)
            cnx.commit()  # Save database changes

            ok += 1
            # Pause briefly to act like a human and avoid overwhelming the server
            sleep_human(1.5, 4.0)

        except Exception as e:
            cnx.rollback()
            fail += 1

            status_code = getattr(resp, "status", None)
            response_size = len(resp.body) if (resp is not None and getattr(resp, "body", None)) else None

            db_exec(cur, """
                INSERT INTO raw_pdf_error (case_id, error_url, status_code, response_size, error_message)
                VALUES (%s, %s, %s, %s, %s)
            """, (str(row_id), pdf_url, status_code, response_size, str(e)[:65535]))
            cnx.commit()

            (DIR_LOG / "pdf_download_errors.log").open("a", encoding="utf-8").write(
                f"{datetime.now().isoformat()} | {pdf_url} | {repr(e)}\n"
            )
            sleep_human(2, 6)

ok, fail